# Student starter
Run cells in order. An incomplete TODO prints guidance and stops subsequent cells. After implementing it, restart the kernel and run again.

# Document Extraction, Cleaning and Chunking
Exercise 10 — Task 11 portion of the RAG notebook. Markdown is canonical; derived PDFs are extraction examples. Embeddings, retrieval, context and generation remain planned. Synthetic educational data created for NLP workshop purposes.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from pathlib import Path
        import sys
        ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
        if not (ROOT / 'src').exists():
            raise RuntimeError('Run from repository root or notebooks/.')
        sys.path.insert(0, str(ROOT))
        from src.document_processor import (load_markdown_document, extract_text_from_pdf, clean_document_text, split_text_into_chunks, create_overlapping_chunks, attach_chunk_metadata, process_document, model_token_lengths)
        from src.sentence_resources import load_sentence_tokenizer
        import json
        manifest = json.loads((ROOT / 'data/document_manifest.json').read_text())
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 1. Load canonical Markdown
Extraction is starter infrastructure. It preserves the raw text; Markdown has no physical page number.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        entry = next(e for e in manifest['academic_documents'] if e['document_id']=='HCE-DOC04')
        document = load_markdown_document(ROOT / 'data' / entry['filename'], document_id=entry['document_id'], title=entry['title'])
        print(document['title'], document['pages'][0]['page_number'])
        print(document['pages'][0]['text'][:600])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 2. Conservative cleaning — TODO 10.1
Normalize whitespace while retaining punctuation, casing, numbers and headings. Do not remove stopwords or stem evidence.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        raw = document['pages'][0]['text']
        cleaned = clean_document_text(raw)
        print(clean_document_text('  Attendance:\t75%.\r\n\r\n\r\nMarks: 40 + 60 = 100! '))
        print(cleaned[:600])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 3. Non-overlapping chunks — TODO 10.2
A whitespace word is not a model token. Keep the last partial chunk.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        small = 'a b c d e f g h'
        print([c['text'] for c in split_text_into_chunks(small, 4)])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 4. Overlap — TODO 10.3
Step = size - overlap. Stop after the final word is covered; do not add an overlap-only tail.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        print([c['text'] for c in create_overlapping_chunks(small, 4, 1)])
        processed = process_document(document)
        print(processed['statistics'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 5. Source metadata — TODO 10.4
IDs are positional and deterministic for unchanged text/settings, not permanent content hashes. Word/character offsets address cleaned text and are zero-based, end-exclusive.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        windows = create_overlapping_chunks(cleaned, 100, 20)
        chunks = attach_chunk_metadata(windows, document)
        print({k:v for k,v in chunks[0].items() if k!='text'})
        assert chunks[0]['text'] == cleaned[chunks[0]['char_start']:chunks[0]['char_end']]
        print(chunks[0]['text'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 6. Derived PDF extraction
Use physical page numbers. Empty physical pages retain their place; page overlap resets. All PDFs are synthetic text documents, not scans. Upload parsing has size/page guardrails and no persistence/OCR.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        pdf_entry = manifest['derived_sample_pdfs'][0]
        pdf = extract_text_from_pdf(ROOT / 'data' / pdf_entry['filename'], document_id=pdf_entry['document_id'], title=pdf_entry['title'])
        print([(p['page_number'], len(p['text'])) for p in pdf['pages']])
        pdf_processed = process_document(pdf)
        print(pdf_processed['statistics'])
        print({k:v for k,v in pdf_processed['chunks'][-1].items() if k!='text'})
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 7. Chunk-size and overlap experiments
Compare counts, repeated word positions and boundary context. This measures preparation, not retrieval quality.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        for size, overlap in [(60,20),(100,0),(100,20),(100,40),(240,20)]:
            result = process_document(document, size, overlap)
            print(size, overlap, result['statistics'])
        print('Shared boundary:', chunks[0]['text'].split()[-20:])
        assert chunks[0]['text'].split()[-20:] == chunks[1]['text'].split()[:20]
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 8. Cached MiniLM tokenizer only
Explicit preflight is required. This loader never downloads or loads encoder weights. Counts are untruncated and include special tokens. If absent, run the existing documented explicit preload before executing this cell.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        tokenizer = load_sentence_tokenizer()
        full_length = model_token_lengths(tokenizer, [cleaned])[0]
        lengths = model_token_lengths(tokenizer, [c['text'] for c in chunks])
        print('Full tokens:', full_length, 'chunk min/max:', min(lengths), max(lengths), 'above 256:', sum(n>256 for n in lengths))
        for size in [60,100,120,240]:
            candidate = process_document(document,size,20)['chunks']
            counts = model_token_lengths(tokenizer,[c['text'] for c in candidate])
            print(size,len(counts),max(counts),sum(n>256 for n in counts))
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 9. Inspect all fourteen canonical documents
Task 09 full-document embeddings could truncate evidence. The measured 100/20 default fits this supplied corpus, not all possible text.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        all_counts=[]
        for e in manifest['academic_documents']:
            d=load_markdown_document(ROOT/'data'/e['filename'],document_id=e['document_id'],title=e['title'])
            p=process_document(d)
            counts=model_token_lengths(tokenizer,[c['text'] for c in p['chunks']])
            all_counts.extend(counts)
            print(e['document_id'],len(counts),max(counts))
        print('Total:',len(all_counts),'max:',max(all_counts),'above256:',sum(n>256 for n in all_counts))
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        attendance = load_markdown_document(ROOT/'data/academic_documents/08_attendance_policy.md')
        policy_line = next(line for line in attendance['pages'][0]['text'].splitlines() if line.startswith('Students must maintain'))
        policy_sentence = policy_line.split('. ')[0] + '.'
        print(policy_sentence)
        for overlap in [0,3]:
            print(overlap, [c['text'] for c in create_overlapping_chunks(policy_sentence,7,overlap)])

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Exercise 11 — Semantic Chunk Retrieval
The retrieval unit changes from whole document to passage. No context, prompt or generation is implemented.

## 10. Canonical chunks and same cached encoder
The benchmark uses only fourteen Markdown sources, never duplicate PDFs or question papers. Keep measured 100/20 defaults. Loading and token preflight are starter code.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from src.retrieval import (load_canonical_documents, build_chunk_corpus, encode_document_chunks, encode_user_question, calculate_chunk_similarities, retrieve_top_k_chunks, semantic_chunk_search)
        from src.sentence_resources import load_sentence_embedding_model
        import numpy as np
        documents = load_canonical_documents(ROOT)
        canonical_chunks = build_chunk_corpus(documents)
        model = load_sentence_embedding_model()
        lengths = model_token_lengths(model.tokenizer, [c['text'] for c in canonical_chunks])
        assert max(lengths)<=model.max_seq_length
        print(len(documents),len(canonical_chunks),max(lengths))
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 11. Encode chunk text — TODO 11.1
One row per ordered passage. Encode original cleaned text, not metadata.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        chunk_matrix = encode_document_chunks(model, canonical_chunks)
        print(chunk_matrix.shape)
        assert chunk_matrix.shape==(152,384)
        print(chunk_matrix[0,:8])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 12. Encode the question — TODO 11.2
Same encoder, one 384-dimensional vector.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        question = 'What is the minimum attendance required?'
        question_vector = encode_user_question(model, question)
        print(len(question_vector),np.linalg.norm(question_vector),question_vector[:8])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 13. Cosine per chunk — TODO 11.3
Reuse existing manual cosine. Preserve chunk-row-score alignment.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        scores = calculate_chunk_similarities(question_vector, chunk_matrix)
        print(len(scores),min(scores),sum(scores)/len(scores),max(scores))
        print(scores[:10])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 14. Top-K evidence — TODO 11.4
Stable descending ranking preserves all provenance. Results are evidence, not answers.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        evidence = retrieve_top_k_chunks(canonical_chunks,scores,5)
        for c in evidence:
            print(c['rank'],c['chunk_id'],c['document_title'],c['score'])
            print(c['text'])
            print({k:v for k,v in c.items() if k!='text'})
        search = semantic_chunk_search(question,canonical_chunks,chunk_matrix,model,5)
        print('Sorted indices:',search['sorted_indices'][:10])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 15. Direct fact, paraphrase, multi-source, comparison and unsupported queries
Judge useful evidence; do not equate nearest passages with relevant facts. Reuse the existing corpus matrix for every question.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        queries = ['In which semester is Natural Language Processing offered?', 'Which subjects must a student complete before taking NLP?', 'When can low attendance stop a student from writing the final examination?', 'What conditions must a student satisfy to complete the NLP course successfully?', 'How are Machine Learning and Deep Learning related in the curriculum?', 'What is the recipe for chocolate cake?']
        for q in queries:
            result = semantic_chunk_search(q,canonical_chunks,chunk_matrix,model,5)
            print(q)
            print([(c['chunk_id'],round(c['score'],4)) for c in result['results']])
            print(result['results'][0]['text'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 16. Overlap redundancy
Adjacent source windows may both rank highly. Core retrieval deliberately does not deduplicate them.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        overlap_search=semantic_chunk_search('What attendance percentage is required?',canonical_chunks,chunk_matrix,model,5)
        for i,left in enumerate(overlap_search['results']):
            for right in overlap_search['results'][i+1:]:
                shared=min(left['word_end'],right['word_end'])-max(left['word_start'],right['word_start'])
                if left['document_id']==right['document_id'] and left['page_number']==right['page_number'] and shared>0:
                    print(left['chunk_id'],right['chunk_id'],'shared positions:',shared)
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 17. Whole-document vs chunk granularity — RET-06
The historical whole-document benchmark used eighteen sources including papers; our canonical chunk corpus uses fourteen academic sources. This comparison is not a controlled truncation-only experiment. Expected references below belong to instructor evaluation only.

In [ ]:
# Instructor measured report omitted. Run your own completed functions to inspect results.


## 18. Recorded instructor evaluation
This diagnostic artifact never supplies live search outputs. Source metrics and evidence checks are separate; expanded and original-case aggregates have different denominators.

In [ ]:
# Instructor measured report omitted. Run your own completed functions to inspect results.


## Exercise 12 — Construct Grounded Context
Keep existing retrieval defaults; add formatting and prompts, with optional generation. Full transparent RAG presentation remains Task14.

## 19. Retrieve Top-5 and preserve evidence
Retrieval gives dictionaries, not an answer. Formatting preserves evidence rather than summarizing it.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from src.rag import (build_context, format_source_block, build_grounded_prompt, prepare_rag_request, run_rag)
        from src.local_llm import RecordingTestLLM
        from src.llm_resources import load_local_llm, local_model_status, LocalModelUnavailable
        attendance_question='What is the minimum attendance required?'
        retrieved=semantic_chunk_search(attendance_question,canonical_chunks,chunk_matrix,model,5)['results']
        print([(c['rank'],c['chunk_id']) for c in retrieved])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 20. Context and labels — TODOs 12.1 / 12.2
Use one SOURCE n label per chunk, preserve order and text. Scores remain metadata; Markdown has no physical page.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        context=build_context(retrieved)
        print(format_source_block(retrieved[1],2))
        print(context)
        prepared=prepare_rag_request(attendance_question,retrieved,requested_top_k=5)
        print(prepared['statistics'])
        print({label:c['chunk_id'] for label,c in prepared['sources'].items()})
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 21. Whole-block context budget
Stop at the first block that cannot fit. Exclusions are reported; no hidden merging/deduplication or mid-block truncation.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        small=prepare_rag_request(attendance_question,retrieved,max_characters=1000,requested_top_k=5)
        print(small['statistics'],small['excluded_chunks'])
        assert len(small['context'])<=1000
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 22. Grounded prompt — TODO 12.3
Instructions + labelled context + question. Documents are evidence, not overriding instructions. These rules request grounding but cannot guarantee it.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        prompt=build_grounded_prompt(attendance_question,context)
        print(prompt)
        assert prompt==prepared['prompt']
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 23. TEST ONLY generation interface
This deterministic adapter records a prompt and returns simulated text. It does not infer answers and is never an app fallback.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        test_llm=RecordingTestLLM('TEST ONLY: simulated answer [SOURCE 2]; no model inference.')
        test_result=run_rag(attendance_question,retrieved,test_llm,requested_top_k=5)
        assert test_llm.prompts[-1]==test_result['prompt']
        print(test_result['answer'])
        print(run_rag(attendance_question,retrieved,None)['generation_status'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 24. Optional actual CPU backend — instructor guided, no algorithm TODO
Preload explicitly before class. If absent, retrieval/context/prompt still execute. Decoder chat templating and token checks are starter infrastructure. Generated text is not verified source evidence.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        print(local_model_status())
        local_backend=None
        try:
            local_backend=load_local_llm()
        except LocalModelUnavailable as error:
            print(error)
        if local_backend is not None:
            actual_result=run_rag(attendance_question,retrieved,local_backend,requested_top_k=5)
            print('GENERATED MODEL OUTPUT:',actual_result['answer'])
            print('Not verified citations:',actual_result['sources'].keys())
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 25. Unsupported question and retrieval failure
Inspect contexts, not expected answers. The syllabus is missed, but lab guidelines provide indirect credit support. Inspect every block; source misses are not necessarily evidence gaps. The cake question has no supported recipe evidence, regardless of nearest-neighbor scores.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        for question in ['What is the recipe for chocolate cake?','How many credits does the NLP course carry?']:
            hits=semantic_chunk_search(question,canonical_chunks,chunk_matrix,model,5)['results']
            request=prepare_rag_request(question,hits,requested_top_k=5)
            print(question,request['included_chunk_ids'])
            print(request['prompt'])
            print('TEST ONLY structural result:',run_rag(question,hits,test_llm)['answer'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 26. K=1/3/5 and weak versus grounded prompts
No query-specific K override. Extra evidence can add facts, noise and duplicates. Generation-model token counts differ from MiniLM counts.

In [ ]:
# Instructor measured report omitted. Run your own completed functions to inspect results.


## Exercise 14 — Complete Transparent RAG
Integrate the existing pipeline and inspect separate retrieval, context, generation and attribution. No new algorithms or prompt tuning.

## 27. One explicit transparent pipeline
The retrieval callback reuses the cached matrix and shared implementation. No-generation mode is complete, not an error.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from src.rag import run_transparent_rag, inspect_source_labels
        retriever=lambda question,k:semantic_chunk_search(question,canonical_chunks,chunk_matrix,model,k)
        transparent=run_transparent_rag(attendance_question,retriever,local_backend,top_k=5,generation_model=local_model_status()['model_id'] if local_backend else None)
        print(transparent.keys())
        print('GENERATED ANSWER:',transparent['generation'])
        print('Context:',transparent['context']['statistics'])
        print('Stage timings:',transparent['timings_seconds'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 28. Independent sources supplied to the model
The list comes from included evidence, even if the model emits no labels. It does not validate every claim.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        for source in transparent['attribution']['supplied_sources']:
            print(source['label'],source['document_id'],source['chunk_id'],source['page_number'],source['rank'])
            print(source['text'])
        print('Model label inspection:',transparent['attribution']['model_label_inspection'])
        print('Diagnostics:',transparent['diagnostics'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 29. Structural labels are not factual support
Known SOURCE 1, unknown SOURCE 99 and no labels are different observations. These checks do not grade claims.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        for text in ['75% [SOURCE 1]','80% [SOURCE 99]','No labels']:
            print(text,inspect_source_labels(text,transparent['context']['included_sources']))
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 30. Retrieval depth and context-budget ablation
Inspect the actual supplied evidence without requiring generation. The model and prompt rules stay fixed.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        attendance_one=run_transparent_rag(attendance_question,retriever,None,top_k=1)
        attendance_five=run_transparent_rag(attendance_question,retriever,None,top_k=5)
        budget_one=run_transparent_rag(attendance_question,retriever,None,top_k=5,max_chunks=1)
        print('K1 context has percentage:', '75%' in attendance_one['context']['text'])
        print('K5 context has percentage:', '75%' in attendance_five['context']['text'])
        print('K5/budget1 excludes:',budget_one['context']['excluded_sources'])
        assert '75%' not in attendance_one['context']['text']
        assert '75%' in attendance_five['context']['text']
        assert '75%' not in budget_one['context']['text']
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 31. Unsupported, multi-source and indirect-support cases
Live pre-generation pipeline; read all blocks. The credit syllabus can be missed while Lab Guidelines supply indirect four-credit theory support.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        for question in ['What is the recipe for chocolate cake?','When can low attendance stop a student from writing the final examination?','How many credits does the NLP course carry?']:
            result=run_transparent_rag(question,retriever,None,top_k=5)
            print(question,result['context']['statistics'])
            print([(s['label'],s['chunk_id']) for s in result['attribution']['supplied_sources']])
            print(result['context']['text'])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## 32. Fresh Task14 measured outputs — not application answers
The evaluator uses production code and separate expected facts. Numbers/phrases are case-specific checks; source-label existence is not entailment.

In [ ]:
# Instructor measured report omitted. Run your own completed functions to inspect results.


## Failure taxonomy and architecture
Unsupported: no answer in the knowledge base. Otherwise inspect useful Top-K evidence (retrieval failure), included context (context/budget failure), generated claim support (generation failure) and source labels (attribution failure). Multiple failures may coexist. The deterministic source-display path originates in included evidence, not the LLM.

Question → embedding → retrieval → context → prompt → optional answer. Included evidence → deterministic source list.

**Checkpoint 11: Complete Transparent RAG Pipeline.** This is educational RAG, not guaranteed factual correctness or deployment readiness. No new algorithm TODOs; 44 remain. Next tasks cover question-paper analysis and final Assistant integration.